# Modeling K-Means Clustering — Kemiskinan Wilayah Indonesia
**Catatan:** Implementasi K-Means dilakukan secara manual tanpa menggunakan `sklearn.cluster.KMeans`.

Pipeline:
1. Load data hasil cleaning
2. Pilih fitur & scaling (MinMaxScaler)
3. Tentukan K optimal → Elbow Method + Silhouette Score (manual)
4. Implementasi K-Means dari nol
5. Evaluasi hasil cluster
6. Visualisasi

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

## 2. Load Data Hasil Cleaning
Ganti path file sesuai lokasi file hasil cleaning dari temanmu.

In [ ]:
df = pd.read_csv('klasifikasi kemiskinan_missing clean.csv')

print('Shape data:', df.shape)
print()
df.head()

## 3. Pilih Fitur
Berdasarkan seleksi dari proses EDA temanmu, 3 fitur ini dipilih karena:
- **Persentase Penduduk Miskin** → indikator langsung kemiskinan
- **Pengeluaran per Kapita** → proxy kesejahteraan ekonomi
- **Umur Harapan Hidup** → proxy kualitas hidup

In [ ]:
features = [
    'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)',
    'Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun)',
    'Umur Harapan Hidup (Tahun)'
]

X = df[features].copy()
print('Fitur yang digunakan:')
X.describe()

## 4. Scaling Manual (Min-Max Normalization)

**Kenapa perlu scaling?**
K-Means menghitung jarak antar titik. Kalau fitur punya skala beda jauh (misalnya % vs ribuan rupiah), fitur dengan angka besar akan mendominasi perhitungan jarak — hasil cluster jadi bias.

**Analogi:** Bayangkan kamu bandingkan berat orang (50–100 kg) dengan tinggi badan (150–180 cm). Kalau langsung dihitung jaraknya, perbedaan berat lebih 'terasa' besar padahal rentangnya mirip. Scaling membuat semua fitur "berbicara dengan volume yang sama".

Formula Min-Max: `X_scaled = (X - X_min) / (X_max - X_min)` → hasil antara 0 dan 1.

In [ ]:
def minmax_scale(X):
    """
    Normalisasi Min-Max secara manual.
    Input : numpy array shape (n_samples, n_features)
    Output: array ternormalisasi, beserta min dan max tiap fitur
    """
    x_min = X.min(axis=0)
    x_max = X.max(axis=0)
    x_scaled = (X - x_min) / (x_max - x_min)
    return x_scaled, x_min, x_max

X_arr = X.values  # ubah ke numpy array
X_scaled, x_min, x_max = minmax_scale(X_arr)

print('Min tiap fitur :', x_min)
print('Max tiap fitur :', x_max)
print()
print('Contoh 5 baris setelah scaling:')
pd.DataFrame(X_scaled, columns=features).head()

## 5. Implementasi K-Means Manual

**Cara kerja K-Means (analogi):**
Bayangkan kamu punya sekumpulan titik di peta dan ingin membaginya jadi K kelompok.
1. Lempar K penanda (centroid) secara acak ke peta
2. Setiap titik "bergabung" ke penanda yang paling dekat
3. Geser setiap penanda ke titik tengah (rata-rata) kelompoknya
4. Ulangi langkah 2–3 sampai penanda tidak bergerak lagi

**Itu saja.** Sederhana, tapi powerful.

In [ ]:
def euclidean_distance(a, b):
    """Hitung jarak Euclidean antara dua titik."""
    return np.sqrt(np.sum((a - b) ** 2))


def kmeans_manual(X, k, max_iter=300, tol=1e-4, random_state=42):
    """
    Implementasi K-Means Clustering dari nol.

    Parameters
    ----------
    X           : numpy array (n_samples, n_features)
    k           : jumlah cluster
    max_iter    : maksimum iterasi
    tol         : toleransi konvergensi (jika pergeseran centroid < tol, berhenti)
    random_state: seed untuk reprodusibilitas

    Returns
    -------
    labels      : array label cluster tiap data point
    centroids   : posisi akhir centroid
    inertia     : total within-cluster sum of squares (WCSS)
    """
    np.random.seed(random_state)
    n_samples = X.shape[0]

    # Langkah 1: Inisialisasi centroid — pilih k titik data secara acak
    idx = np.random.choice(n_samples, k, replace=False)
    centroids = X[idx].copy()

    labels = np.zeros(n_samples, dtype=int)

    for iteration in range(max_iter):

        # Langkah 2: Assign setiap titik ke centroid terdekat
        new_labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            distances = [euclidean_distance(X[i], centroids[c]) for c in range(k)]
            new_labels[i] = np.argmin(distances)

        # Langkah 3: Hitung centroid baru (rata-rata tiap cluster)
        new_centroids = np.zeros_like(centroids)
        for c in range(k):
            members = X[new_labels == c]
            if len(members) == 0:
                # Kalau cluster kosong, reinisialisasi centroid ke titik acak
                new_centroids[c] = X[np.random.randint(n_samples)]
            else:
                new_centroids[c] = members.mean(axis=0)

        # Langkah 4: Cek konvergensi — apakah centroid masih bergerak?
        shift = np.max([euclidean_distance(new_centroids[c], centroids[c]) for c in range(k)])

        labels = new_labels
        centroids = new_centroids

        if shift < tol:
            print(f'  Konvergen pada iterasi ke-{iteration + 1} (pergeseran centroid = {shift:.6f})')
            break

    # Hitung inertia (WCSS)
    inertia = 0
    for i in range(n_samples):
        inertia += euclidean_distance(X[i], centroids[labels[i]]) ** 2

    return labels, centroids, inertia


print('Fungsi K-Means manual siap digunakan.')

## 6. Elbow Method — Menentukan K Optimal

**Cara baca:** Kita jalankan K-Means untuk K = 1 hingga 10, lalu plot nilai **inertia** (total jarak tiap titik ke centroid clusternya).
Di titik di mana penurunan inertia mulai melambat drastis (seperti bentuk siku/elbow), itulah K yang paling optimal — menambah cluster lebih banyak tidak memberi manfaat signifikan.

In [ ]:
inertia_list = []
K_range = range(1, 11)

print('Menjalankan Elbow Method...')
for k in K_range:
    print(f'K = {k}', end=' ')
    _, _, inertia = kmeans_manual(X_scaled, k=k, random_state=42)
    inertia_list.append(inertia)
    print(f'→ Inertia: {inertia:.4f}')

plt.figure(figsize=(8, 5))
plt.plot(list(K_range), inertia_list, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method — Menentukan K Optimal')
plt.xticks(list(K_range))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Silhouette Score Manual — Konfirmasi K Optimal

**Apa itu Silhouette Score?**
Ukuran seberapa 'pas' sebuah titik berada di clusternya.
- Nilainya antara -1 sampai 1
- Mendekati 1 → titik sudah di cluster yang tepat
- Mendekati 0 → titik ada di perbatasan antara 2 cluster
- Negatif → titik kemungkinan salah cluster

**Analogi:** Kamu pindah ke kota baru dan bergabung komunitas. Silhouette Score = seberapa cocok kamu dengan komunitas itu dibanding komunitas lain yang ada.

In [ ]:
def silhouette_score_manual(X, labels):
    """
    Hitung Silhouette Score secara manual.
    
    Untuk setiap titik i:
    - a(i) = rata-rata jarak ke semua titik lain DALAM cluster yang sama
    - b(i) = rata-rata jarak ke titik di cluster TERDEKAT lain
    - s(i) = (b(i) - a(i)) / max(a(i), b(i))

    Silhouette Score = rata-rata s(i) semua titik
    """
    n = X.shape[0]
    unique_clusters = np.unique(labels)
    s_scores = []

    for i in range(n):
        own_cluster = labels[i]

        # a(i): rata-rata jarak ke sesama anggota cluster sendiri
        own_members = X[labels == own_cluster]
        if len(own_members) == 1:
            s_scores.append(0)
            continue
        a_i = np.mean([euclidean_distance(X[i], own_members[j])
                       for j in range(len(own_members)) if not np.array_equal(own_members[j], X[i])])

        # b(i): rata-rata jarak ke cluster terdekat
        b_i = np.inf
        for c in unique_clusters:
            if c == own_cluster:
                continue
            other_members = X[labels == c]
            avg_dist = np.mean([euclidean_distance(X[i], other_members[j])
                                for j in range(len(other_members))])
            if avg_dist < b_i:
                b_i = avg_dist

        s_i = (b_i - a_i) / max(a_i, b_i)
        s_scores.append(s_i)

    return np.mean(s_scores)


print('Menghitung Silhouette Score untuk K=2 s/d 6...')
print('(Catatan: perhitungan manual membutuhkan waktu lebih lama dari library)\n')

sil_scores = {}
for k in range(2, 7):
    labels_temp, _, _ = kmeans_manual(X_scaled, k=k, random_state=42)
    score = silhouette_score_manual(X_scaled, labels_temp)
    sil_scores[k] = score
    print(f'K={k}  →  Silhouette Score: {score:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(list(sil_scores.keys()), list(sil_scores.values()), marker='s', color='coral', linewidth=2)
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score tiap K')
plt.xticks(list(sil_scores.keys()))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

best_k = max(sil_scores, key=sil_scores.get)
print(f'\nK terbaik berdasarkan Silhouette Score: K = {best_k} (score = {sil_scores[best_k]:.4f})')

## 8. Training Model Final
Gunakan K optimal yang sudah ditemukan dari langkah sebelumnya.

In [ ]:
K_OPTIMAL = best_k  # sesuaikan jika ingin override manual

print(f'Training K-Means dengan K = {K_OPTIMAL}...')
labels_final, centroids_final, inertia_final = kmeans_manual(
    X_scaled, k=K_OPTIMAL, random_state=42
)

df['Cluster'] = labels_final

print(f'\nInertia final : {inertia_final:.4f}')
print(f'Distribusi cluster:')
print(df['Cluster'].value_counts().sort_index())

## 9. Interpretasi Cluster
Lihat rata-rata fitur tiap cluster untuk memahami karakteristiknya.

In [ ]:
cluster_summary = df.groupby('Cluster')[features].mean().round(2)
print('Rata-rata fitur per cluster:')
print(cluster_summary)
print()

# Beri label berdasarkan nilai Persentase Penduduk Miskin (semakin tinggi = semakin miskin)
poverty_rank = cluster_summary['Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)'].rank(ascending=False)
label_map = {}
rank_labels = ['Sangat Miskin', 'Miskin', 'Cukup', 'Sejahtera']  # sesuaikan jika K berbeda

for cluster_id, rank in poverty_rank.items():
    idx = int(rank) - 1
    if idx < len(rank_labels):
        label_map[cluster_id] = rank_labels[idx]
    else:
        label_map[cluster_id] = f'Cluster {cluster_id}'

df['Kategori'] = df['Cluster'].map(label_map)

print('Mapping label cluster:')
for c, lbl in label_map.items():
    print(f'  Cluster {c} → {lbl}')

## 10. Visualisasi Hasil Clustering

In [ ]:
colors = cm.get_cmap('Set1', K_OPTIMAL)

plt.figure(figsize=(9, 6))
for c in range(K_OPTIMAL):
    mask = labels_final == c
    plt.scatter(
        X_scaled[mask, 0],
        X_scaled[mask, 1],
        label=f'Cluster {c} — {label_map.get(c, "")}',
        alpha=0.7,
        s=50
    )

# Plot centroid
plt.scatter(
    centroids_final[:, 0],
    centroids_final[:, 1],
    marker='X', s=200, color='black', zorder=5, label='Centroid'
)

plt.xlabel('Persentase Penduduk Miskin (scaled)')
plt.ylabel('Pengeluaran per Kapita (scaled)')
plt.title(f'Hasil K-Means Manual (K={K_OPTIMAL})')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 11. Sample Hasil per Cluster

In [ ]:
for c in sorted(df['Cluster'].unique()):
    print(f'\n===== CLUSTER {c} — {label_map.get(c, "")} =====')
    subset = df[df['Cluster'] == c][['Provinsi', 'Kab/Kota', 'Kategori'] + features]
    print(f'Jumlah wilayah: {len(subset)}')
    print(subset.head(10).to_string(index=False))

## 12. Simpan Hasil untuk Geospatial Visualizer

In [ ]:
output_cols = ['Provinsi', 'Kab/Kota', 'Cluster', 'Kategori'] + features
df_out = df[output_cols].copy()

df_out.to_csv('hasil_clustering_kemiskinan.csv', index=False)
print('File tersimpan: hasil_clustering_kemiskinan.csv')
print(f'Total baris: {len(df_out)}')
df_out.head()